# FastSpeech2 — Bulgarian single-speaker (Euthymius) on Colab

All-in-one pipeline: **setup → MFA alignment → preprocess → train → synthesize**.

**Before you start**
1. Runtime → *Change runtime type* → **GPU** (T4 is fine).
2. Zip your dataset folder so it becomes `Euthymius.zip` (containing `reaper_0806_librivox/` and `staroplaninski_legendi_eu_0812_librivox-1/`) and upload it to **MyDrive** (Drive root).
3. Run the cells top to bottom. Cell **1 restarts the runtime** — after it restarts, continue from section 2.

In [ ]:
!nvidia-smi

## 1. Install MFA via condacolab
⚠️ This **restarts the runtime automatically**. After the restart, *skip this cell* and continue from section 2.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()  # runtime restarts here

## 2. Install MFA + Python dependencies *(run after the restart)*
The `conda install` of MFA pulls Kaldi and can take a few minutes.

In [ ]:
import condacolab
condacolab.check()
# Montreal Forced Aligner (Kaldi-based; conda only)
!conda install -y -c conda-forge montreal-forced-aligner
# FastSpeech2 Python deps (modern versions; the repo code was patched for them)
!pip install -q torch torchaudio numpy scipy librosa pyworld tgt scikit-learn matplotlib tensorboard pyyaml tqdm soundfile

## 3. Mount Google Drive
Used for the dataset and for persistent checkpoints/logs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone the project

In [ ]:
%cd /content
!rm -rf FastSpeech2
!git clone https://github.com/StankoI/FastSpeech2.git
%cd /content/FastSpeech2

## 5. Unpack the dataset from Drive
Unzips to `Euthymius/...` at the repo root, which is where the cleaned manifest's relative paths point.

In [ ]:
!cp "/content/drive/MyDrive/Euthymius.zip" .
!unzip -q -o Euthymius.zip
import os
print('manifest exists:', os.path.exists('filelists/euthymius_clean.csv'))
first = open('filelists/euthymius_clean.csv', encoding='utf-8').readline().split('|')[1]
print('first wav resolves:', first, '->', os.path.exists(first))

## 6. Unzip the HiFi-GAN universal vocoder
The loader expects `hifigan/generator_universal.pth.tar` (the repo ships it zipped).

In [ ]:
!unzip -o hifigan/generator_universal.pth.tar.zip -d hifigan/
!ls -la hifigan/*.pth.tar

## 7. prepare_align — resample to 22.05 kHz and write `.wav`/`.lab` for MFA

In [ ]:
!python prepare_align.py config/Bulgarian/preprocess.yaml
!ls raw_data/Bulgarian/Bulgarian | head

## 8. MFA — train a grapheme aligner from scratch and export TextGrids
The slow step (~10–40 min). Our 'phones' are Cyrillic letters, so MFA's pretrained IPA model is unusable — we train one on this corpus and export the alignments to `preprocessed_data/Bulgarian/TextGrid/`.

*If MFA errors on its database backend, it usually just needs `--clean`; a re-run of this cell often succeeds.*

In [ ]:
!mfa validate raw_data/Bulgarian lexicon/bulgarian-grapheme.txt --clean -j 4 || true
!mfa train raw_data/Bulgarian lexicon/bulgarian-grapheme.txt bg_acoustic_model.zip --output_directory preprocessed_data/Bulgarian/TextGrid --clean -j 4
!ls preprocessed_data/Bulgarian/TextGrid/Bulgarian | head

In [ ]:
# === Save step 7–8 outputs to Drive (run after MFA finishes) ===
import os, shutil

BK = "/content/drive/MyDrive/fs2_bg"
os.makedirs(BK, exist_ok=True)

# 1) MFA alignments — the slow, valuable result
assert os.path.isdir("preprocessed_data/Bulgarian/TextGrid"), \
    "No TextGrids found — did step 8 finish successfully?"
shutil.make_archive(os.path.join(BK, "TextGrid"), "zip",
                    "preprocessed_data/Bulgarian", "TextGrid")

# 2) Trained MFA acoustic model
if os.path.exists("bg_acoustic_model.zip"):
    shutil.copy("bg_acoustic_model.zip", BK)

# 3) Resampled corpus (so preprocess can run on resume without redoing step 7)
shutil.make_archive(os.path.join(BK, "raw_data_Bulgarian"), "zip",
                    "raw_data", "Bulgarian")

print("Saved to", BK)
for f in sorted(os.listdir(BK)):
    p = os.path.join(BK, f)
    if os.path.isfile(p):
        print(f"  {f:28s} {os.path.getsize(p)/1e6:8.1f} MB")

### (optional) Back up the alignments + acoustic model to Drive
So a disconnect doesn't force you to re-run MFA.

In [ ]:
!mkdir -p /content/drive/MyDrive/fs2_bg
!cp bg_acoustic_model.zip /content/drive/MyDrive/fs2_bg/
!cd preprocessed_data/Bulgarian && zip -q -r /content/drive/MyDrive/fs2_bg/TextGrid.zip TextGrid
print('backed up to MyDrive/fs2_bg')

## 9. Preprocess — mel / pitch / energy / duration features
Runs on GPU (the STFT uses CUDA). Produces `train.txt`, `val.txt`, `stats.json`, `speakers.json`.

In [ ]:
!python preprocess.py config/Bulgarian/preprocess.yaml
!wc -l preprocessed_data/Bulgarian/train.txt preprocessed_data/Bulgarian/val.txt

## 10. Persist checkpoints to Drive
Symlink `output/` into Drive so checkpoints + logs survive a runtime disconnect (resume with `--restore_step`).

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/fs2_bg/output', exist_ok=True)
!rm -rf output
!ln -s /content/drive/MyDrive/fs2_bg/output output
!ls -la output

## 11. Train
Checkpoints land in Drive every 10k steps. First listenable samples usually appear within a few thousand steps (see TensorBoard).

**Resuming after a disconnect:** re-run sections 2–6 + 10, then add `--restore_step <N>` to the command below.

In [ ]:
!python train.py -p config/Bulgarian/preprocess.yaml -m config/Bulgarian/model.yaml -t config/Bulgarian/train.yaml

In [ ]:
!MPLBACKEND=Agg python train.py -p config/Bulgarian/preprocess.yaml -m config/Bulgarian/model.yaml -t config/Bulgarian/train.yaml
#for colab

## 12. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir output/log/Bulgarian

## 13. Synthesize
Generate speech from text using a saved checkpoint (replace `10000` with the step you want):

In [ ]:
!python synthesize.py --text "Здравей, как си днес?" --restore_step 10000 --mode single -p config/Bulgarian/preprocess.yaml -m config/Bulgarian/model.yaml -t config/Bulgarian/train.yaml

In [ ]:
# to restore existing model from drive
import os, shutil
BK = "/content/drive/MyDrive/fs2_bg"
os.makedirs("preprocessed_data/Bulgarian", exist_ok=True)
shutil.unpack_archive(f"{BK}/raw_data_Bulgarian.zip", "raw_data")
shutil.unpack_archive(f"{BK}/TextGrid.zip", "preprocessed_data/Bulgarian")
print("ok:", os.path.isdir("raw_data/Bulgarian"),
      os.path.isdir("preprocessed_data/Bulgarian/TextGrid/Bulgarian"))

In [ ]:
!python preprocess.py config/Bulgarian/preprocess.yaml
!ls preprocessed_data/Bulgarian/stats.json

In [ ]:
# for colab
!MPLBACKEND=Agg python synthesize.py --text "Здравей! Как си днес?" --restore_step 20000 --mode single -p config/Bulgarian/preprocess.yaml -m config/Bulgarian/model.yaml -t config/Bulgarian/train.yaml